[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/05_data_mixtures/05_data_mixtures.ipynb)

# 05 · 数据配比与合成数据（DoReMi、重复递减、模型崩溃）

目标：用纯 numpy 把配比建模成优化问题，实现 **DoReMi 式 group-DRO 配比优化**、**重复 epoch 收益递减曲线**、**data-constrained scaling 外推**、**model collapse 模拟**。

路线：配比=单纯形权重 → group-DRO 自动配比(核心) → 重复 epoch 有效数据曲线 → data-constrained scaling → model collapse → ✏️ 练习(混合权重优化 / 重复收益 / scaling 外推 / 去重对配比影响) → 📖 答案 → 🧪 真实配比算账胶囊。

> 心智模型：**配比=和为1的权重；DoReMi=给最差 domain 加权拉平 loss；数据墙=重复边际递减；崩溃=有限采样逐代丢尾部**。

In [ ]:
import numpy as np
import math
rng = np.random.default_rng(0)
print('环境就绪，开始配数据')

## 1 · 配比 = 概率单纯形上的权重

配比是和为 1 的权重向量。给定各 domain 单独的 loss，加权平均 loss 随配比变化。先建立这个基本对象。

In [ ]:
domains = ['web', 'code', 'books', 'multilingual']
per_domain_loss = np.array([2.8, 2.2, 4.0, 3.1])   # 各 domain 当前 loss(code 易/books 难)

def is_valid_mixture(w):
    w = np.asarray(w, dtype=float)
    return np.isclose(w.sum(), 1.0) and (w >= 0).all()

def avg_loss(w, losses):
    return float(np.asarray(w) @ losses)

uniform = np.ones(4) / 4
web_heavy = np.array([0.7, 0.1, 0.1, 0.1])
balanced_hard = np.array([0.2, 0.1, 0.5, 0.2])   # 给难的 books 多权重

for name, w in [('均匀', uniform), ('偏 web', web_heavy), ('偏难(books)', balanced_hard)]:
    print(f'{name:12s} 配比 {np.round(w,2)} -> 加权 loss {avg_loss(w, per_domain_loss):.3f}')

assert is_valid_mixture(uniform) and is_valid_mixture(web_heavy)
assert not is_valid_mixture([0.5, 0.6, 0, 0]), '和≠1 应非法'
assert np.isclose(avg_loss(uniform, per_domain_loss), per_domain_loss.mean())
print('\n✅ 配比是单纯形上的权重；加权 loss 随配比变化 —— 这是配比优化的目标函数')

## 2 · DoReMi：group-DRO 自动找配比（本模块核心）

**group-DRO 最小化最差 domain 的 loss**(而非平均)。用乘法权重：给当前 loss 高的 domain 加权，归一化。

关键：domain 的 loss **依赖它的权重**(权重高→学得好→loss 低)。我们建模 `loss_k(w_k) = base_k·exp(-2·w_k)+floor`，group-DRO 会自动**拉平各 domain loss**。

In [ ]:
base_difficulty = np.array([3.0, 2.0, 4.0, 2.5])   # 各 domain 内在难度(books 最难)
FLOOR = 1.0

def domain_losses(w):
    '''domain loss 随其权重下降(更多权重->学得好)，但有下限。'''
    return base_difficulty * np.exp(-2.0 * np.maximum(w, 0)) + FLOOR

def doremi_weights(eta=0.3, steps=300):
    '''group-DRO 乘法权重：上调当前 loss 高的 domain，拉平各 domain loss。'''
    w = np.ones(len(base_difficulty)) / len(base_difficulty)
    history = [w.copy()]
    for _ in range(steps):
        losses = domain_losses(w)
        w = w * np.exp(eta * (losses - losses.mean()))   # 高 loss -> 加权
        w = w / w.sum()                                   # 归一化回单纯形
        history.append(w.copy())
    return w, history

w_uniform = np.ones(4) / 4
w_doremi, hist = doremi_weights()
print('均匀配比 :', np.round(w_uniform, 3), '-> domain loss', np.round(domain_losses(w_uniform), 3))
print('DoReMi   :', np.round(w_doremi, 3), '-> domain loss', np.round(domain_losses(w_doremi), 3))
print(f'\n最差 domain loss: 均匀 {domain_losses(w_uniform).max():.3f} -> DoReMi {domain_losses(w_doremi).max():.3f}')

# DoReMi: 1)给最难 domain(idx2) 加权 2)拉平各 domain loss 3)降低最差 domain loss 4)解在内部(无 domain 归零)
assert w_doremi[2] > w_uniform[2], 'DoReMi 应给最难 domain(books)更多权重'
assert domain_losses(w_doremi).max() < domain_losses(w_uniform).max(), 'DoReMi 应降低最差 domain loss'
spread_u = domain_losses(w_uniform).max() - domain_losses(w_uniform).min()
spread_d = domain_losses(w_doremi).max() - domain_losses(w_doremi).min()
assert spread_d < spread_u, 'DoReMi 应拉平各 domain loss(spread 变小)'
assert (w_doremi > 0.01).all(), 'DoReMi 解在单纯形内部(不抛弃任何 domain)'
print('✅ DoReMi: 自动给难 domain 加权、拉平各 domain loss、不抛弃任何 domain —— 无需下游标签')

> 这就是 DoReMi 的核心：**不优化平均(会被易 domain 主导)，而优化最差 domain**。乘法权重让难学的 domain 自动获得更多采样，最终各 domain loss 被「拉平」——一个均衡、鲁棒的配比。真实 DoReMi 在小 proxy 上跑这个过程，把平均权重迁到大模型。

## 3 · 数据墙：重复 epoch 的有效数据量曲线

数据不够要重复(多 epoch)。Muennighoff 2023：约 4 epoch 内重复≈新数据，之后收益迅速衰减。有效数据 `D_eff(R) = D·τ·(1-exp(-R/τ))`，τ≈4。

In [ ]:
def effective_data(D, R, tau=4.0):
    '''重复 R 个 epoch 的有效数据量(带饱和)。'''
    return D * tau * (1.0 - math.exp(-R / tau))

D = 100.0   # 独特数据量
print(f"{'epoch R':>8} {'有效数据':>10} {'vs 理想 R·D':>12} {'本轮新增':>10}")
prev = 0.0
for R in [1, 2, 4, 8, 16, 32]:
    eff = effective_data(D, R)
    ideal = R * D
    print(f'{R:8d} {eff:10.1f} {eff/ideal:11.1%} {eff-prev:10.1f}')
    prev = eff

# 前几个 epoch 接近理想(重复≈新数据)，后面迅速衰减、饱和
assert effective_data(D, 1) / (1 * D) > 0.85, '第1 epoch 有效率高(≈新数据)'
assert effective_data(D, 32) / (32 * D) < 0.15, '32 epoch 时有效率很低(重复无用)'
assert effective_data(D, 1000) < D * 4.0 + 1e-6, '无限重复饱和到 D·τ'
# 边际递减(等间隔)
marg = [effective_data(D, R+1) - effective_data(D, R) for R in [1,2,3,4,5]]
assert all(marg[i] > marg[i+1] for i in range(len(marg)-1)), '每多一个 epoch 新增递减'
print('\n✅ 重复 epoch 有效数据递增但边际递减、最终饱和 —— 这就是数据墙的数学形状')

## 4 · data-constrained scaling：外推数据受限区的 loss

把 Chinchilla 公式里的 D 换成有效数据 D_eff，就能外推「重复 R epoch + 模型 N」能到的 loss。看重复的 loss 改善如何提前压平。

In [ ]:
def loss_scaling(N, D_eff, E=1.5, A=400.0, B=400.0, alpha=0.34, beta=0.28):
    '''Chinchilla 形式的 loss(用有效数据 D_eff)。常数为示意量级。'''
    return E + A / N**alpha + B / D_eff**beta

N = 1e9          # 固定模型大小
D_unique = 1e10  # 独特 token 数
print('固定 N=1B, 独特数据 10B token，重复不同 epoch:')
print(f"{'epoch R':>8} {'D_eff':>12} {'预测 loss':>10}")
prev_loss = None
for R in [1, 2, 4, 8, 16]:
    D_eff = effective_data(D_unique, R)
    L = loss_scaling(N, D_eff)
    print(f'{R:8d} {D_eff:12.2e} {L:10.4f}')
    prev_loss = L

# loss 随重复下降但改善递减(因 D_eff 饱和)
L1 = loss_scaling(N, effective_data(D_unique, 1))
L4 = loss_scaling(N, effective_data(D_unique, 4))
L16 = loss_scaling(N, effective_data(D_unique, 16))
assert L4 < L1, '重复应降 loss'
assert (L1 - L4) > (L4 - L16), 'loss 改善边际递减(1->4 比 4->16 改善大)'
print('\n✅ data-constrained scaling: 重复降 loss 但改善提前压平(D_eff 饱和所致)')

## 5 · model collapse：合成数据递归训练的崩溃

反复「用模型生成数据→训新模型」= 反复有限采样。有限样本**系统性低估方差**，几代后分布尾部丢失、方差塌缩。模拟之。

In [ ]:
def model_collapse(generations=12, n_samples=40, seed=0):
    '''每代用上一代分布的 n 个样本重新估计(MLE)分布参数，看方差如何衰减。'''
    g = np.random.default_rng(seed)
    mu, sigma = 0.0, 1.0
    stds = [sigma]
    for _ in range(generations):
        samples = g.normal(mu, sigma, size=n_samples)   # 上一代「生成」数据
        mu, sigma = samples.mean(), samples.std()        # 重新拟合(有限样本->有偏)
        stds.append(sigma)
    return np.array(stds)

# 小样本崩溃快，大样本崩溃慢
stds_small = model_collapse(generations=12, n_samples=20, seed=1)
stds_large = model_collapse(generations=12, n_samples=500, seed=1)
print('每代 std (n=20 小样本):', np.round(stds_small[:8], 3))
print('每代 std (n=500 大样本):', np.round(stds_large[:8], 3))
print(f'\n12 代后 std: n=20 -> {stds_small[-1]:.3f} | n=500 -> {stds_large[-1]:.3f} (初始 1.0)')

# 递归训练使方差塌缩；样本越少塌缩越快
assert stds_small[-1] < stds_small[0], '递归训练使方差塌缩(model collapse)'
assert stds_small[-1] < stds_large[-1], '样本越少崩溃越快'
print('✅ model collapse: 纯合成递归训练使方差/多样性塌缩 —— 故合成数据须与真实数据混合')

---
## ✏️ 练习 1：混合权重优化（最小化最差 domain）

实现 `worst_domain_loss(w, base, floor=1.0)`：用第 2 节的 loss 模型 `base·exp(-2w)+floor`，返回当前配比下**最差 domain 的 loss**。

这是 group-DRO 要最小化的目标。

In [ ]:
def worst_domain_loss(w, base, floor=1.0):
    # TODO: losses = base*exp(-2*w)+floor; 返回 max(losses)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
base = np.array([3.0, 2.0, 4.0, 2.5])
w_uni = np.ones(4) / 4
wl_uni = worst_domain_loss(w_uni, base)
print(f'均匀配比的最差 domain loss = {wl_uni:.3f}')
# DoReMi 配比的最差 loss 应更低
wl_doremi = worst_domain_loss(w_doremi, base)
print(f'DoReMi 配比的最差 domain loss = {wl_doremi:.3f}')
assert wl_doremi < wl_uni, 'DoReMi 应降低最差 domain loss'
assert np.isclose(wl_uni, (base * np.exp(-2*w_uni) + 1.0).max())
print('✅ 练习 1 通过：会算 group-DRO 的优化目标(最差 domain loss)')

## ✏️ 练习 2：重复 epoch 的收益率

实现 `repeat_efficiency(D, R, tau=4.0)`：返回重复 R epoch 的**有效数据 ÷ 理想数据(R·D)**，即「这一配置下重复的有效率」。

应随 R 单调下降(重复越多越不划算)。

In [ ]:
def repeat_efficiency(D, R, tau=4.0):
    # TODO: effective_data(D,R,tau) / (R*D)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
effs = [repeat_efficiency(100, R) for R in [1, 2, 4, 8, 16]]
print('R=1,2,4,8,16 的重复有效率:', [round(e, 3) for e in effs])
assert effs[0] > 0.85, '第1 epoch 有效率高'
assert all(effs[i] > effs[i+1] for i in range(len(effs)-1)), '有效率应随 R 单调下降'
assert effs[-1] < 0.4, '16 epoch 时有效率已很低'
print('✅ 练习 2 通过：量化「重复越多越不划算」')

## ✏️ 练习 3：重复的 loss 改善会饱和

data-constrained scaling 的核心后果：**重复 epoch 对 loss 的改善会饱和**(因有效数据饱和)。

实现 `loss_at_epochs(N, D_unique, R)`：返回模型 N、独特数据 D_unique、重复 R epoch 的预测 loss(复用 `loss_scaling` / `effective_data`)。然后验证：**从 4→8 epoch 的 loss 改善，远小于 1→4 epoch**(过了 ~τ 个 epoch 后，再重复几乎无用)。

In [ ]:
def loss_at_epochs(N, D_unique, R):
    # TODO: loss_scaling(N, effective_data(D_unique, R))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
N, D = 1e9, 1e10
L1 = loss_at_epochs(N, D, 1)
L4 = loss_at_epochs(N, D, 4)
L8 = loss_at_epochs(N, D, 8)
L32 = loss_at_epochs(N, D, 32)
print(f'loss @ 1/4/8/32 epoch: {L1:.4f} / {L4:.4f} / {L8:.4f} / {L32:.4f}')
gain_1to4 = L1 - L4         # 前 4 epoch 的 loss 改善
gain_4to8 = L4 - L8         # 第 4-8 epoch 的改善
print(f'loss 改善: 1->4 epoch = {gain_1to4:.4f} | 4->8 epoch = {gain_4to8:.4f}')
assert L4 < L1 and L8 < L4, '重复应降 loss'
assert gain_4to8 < gain_1to4 / 3, '过了 ~τ epoch 后重复的改善大幅缩水(饱和)'
assert (L8 - L32) < gain_4to8, '8->32 epoch 改善更小(几乎榨干)'
print('✅ 练习 3 通过：重复的 loss 改善随 epoch 饱和 —— 故数据受限时该把算力转向加参数/找新数据')

## ✏️ 练习 4：去重对配比的影响

去重(模块 01)减少每个 domain 的**独特** token。实现 `effective_mixture(raw_tokens, dup_rates, target_total)`：
- 各 domain 去重后独特量 = `raw_tokens * (1 - dup_rate)`；
- 返回去重后各 domain 占总量的**实际配比**(归一化)。

看清：去重率不同会**改变**实际配比(重复多的 domain 占比被压低)。

In [ ]:
def effective_mixture(raw_tokens, dup_rates, target_total=None):
    # TODO: unique = raw_tokens*(1-dup_rates); 返回 unique/unique.sum()
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
raw = np.array([1000.0, 1000.0, 1000.0])     # 三 domain 原始量相同
dup = np.array([0.8, 0.2, 0.5])              # web 重复多(0.8)，code 重复少(0.2)
eff_mix = effective_mixture(raw, dup)
print('原始配比: [0.33, 0.33, 0.33]')
print('去重后实际配比:', np.round(eff_mix, 3))
assert np.isclose(eff_mix.sum(), 1.0), '配比和为1'
# 重复最多的 domain(web, idx0) 去重后占比被压低；重复最少的(code, idx1)占比升高
assert eff_mix[0] < 1/3, '重复多的 domain 去重后占比下降'
assert eff_mix[1] > 1/3, '重复少的 domain 去重后占比上升'
assert eff_mix[1] > eff_mix[0], 'code(少重复)应比 web(多重复)占比高'
print('✅ 练习 4 通过：去重会改变实际配比 —— 清洗与配比必须一起考虑')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def worst_domain_loss(w, base, floor=1.0):
    losses = base * np.exp(-2.0 * np.maximum(w, 0)) + floor
    return float(losses.max())

In [ ]:
# 练习 2 参考答案
def repeat_efficiency(D, R, tau=4.0):
    return effective_data(D, R, tau) / (R * D)

In [ ]:
# 练习 3 参考答案
def loss_at_epochs(N, D_unique, R):
    return loss_scaling(N, effective_data(D_unique, R))

In [ ]:
# 练习 4 参考答案
def effective_mixture(raw_tokens, dup_rates, target_total=None):
    unique = np.asarray(raw_tokens) * (1 - np.asarray(dup_rates))
    return unique / unique.sum()

---
## 🧪 真实数据胶囊：LLaMA 风格数据配比算账

用接近 LLaMA / RedPajama 公开的 domain 配比，算一个 1.4T token 预算下各 domain 实际 token 数，并看若某 domain 数据不足需重复多少 epoch。

In [ ]:
# 接近 LLaMA / RedPajama 的 domain 配比(近似公开值)
mixture = {
    'CommonCrawl':  0.67,
    'C4':           0.15,
    'GitHub(code)': 0.045,
    'Wikipedia':    0.045,
    'Books':        0.045,
    'ArXiv':        0.025,
    'StackExchange':0.02,
}
total_budget = 1.4e12     # 1.4T token

weights = np.array(list(mixture.values()))
assert np.isclose(weights.sum(), 1.0), '配比应和为1'
print(f"{'domain':16s} {'权重':>6s} {'token 数':>12s}")
for name, w in mixture.items():
    print(f'{name:16s} {w:6.3f} {w*total_budget:12.2e}')

# 假设 ArXiv 实际只有这么多独特 token，看需重复几个 epoch
arxiv_tokens_needed = mixture['ArXiv'] * total_budget
arxiv_unique_available = 2.0e10   # 假设只有 20B 独特 token
epochs_needed = arxiv_tokens_needed / arxiv_unique_available
print(f'\nArXiv 需要 {arxiv_tokens_needed:.2e} token, 独特只有 {arxiv_unique_available:.2e}')
print(f'-> 需重复约 {epochs_needed:.1f} 个 epoch')
assert epochs_needed > 1, '小 domain 常需重复(数据墙的现实)'
print('✅ 胶囊：真实配比下，小而珍贵的 domain(ArXiv/Books)常需重复多 epoch')

**🧪 胶囊练习**：实现 `epochs_for_domain(weight, total_budget, unique_available)`：返回某 domain 在给定配比下需重复多少 epoch。

In [ ]:
def epochs_for_domain(weight, total_budget, unique_available):
    # TODO: (weight*total_budget) / unique_available
    raise NotImplementedError

In [ ]:
# 自测
e = epochs_for_domain(0.025, 1.4e12, 2.0e10)
print(f'ArXiv 需重复 {e:.1f} epoch')
assert abs(e - 1.75) < 0.01
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def epochs_for_domain(weight, total_budget, unique_available):
    return (weight * total_budget) / unique_available

### 小结
- **配比 = 单纯形权重**，决定模型把有限容量花在学什么上；是个高维、互相牵制、评估昂贵的优化问题。
- **DoReMi (group-DRO)**：不优化平均(被易 domain 主导)，而优化**最差 domain**，乘法权重自动给难 domain 加权、拉平各 domain loss；小代理上跑、迁到大模型。
- **数据墙**：重复 epoch 有效数据递增但边际递减(约 4 epoch 内≈新数据，之后衰减)；data-constrained scaling 把它算进 loss 外推，权衡「重复 vs 加参数」。
- **合成数据**：能补稀缺/提质，但纯递归自训会 **model collapse**(有限采样逐代丢尾部、方差塌缩)；须与真实数据混合、严控比例。
- **课程/质量退火**：末期切高质量数据 + 小 lr 精雕，与 WSD 调度天然配合。

🎉 **全课完成！** 你已走完从 CommonCrawl 到 checkpoint 的完整预训练流水线：清洗去重 → tokenizer → 配比 → μP 超参 → 稳定性。回到 [课程主页](../index.html) 复习，或带着这套 numpy 验证过的算法，去读 FineWeb / Chinchilla / μP / DoReMi 的原论文。